# Image Classification with a Small CNN

Train a compact CNN on CIFAR-10 using PyTorch. First run downloads CIFAR-10. Google Colab GPU is recommended.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
import torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt
SEED=42
torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)

## 1. Load CIFAR-10

Images are 32×32 RGB images in ten classes. Normalize channels to approximately [-1, 1].

In [ ]:
transform=T.Compose([T.ToTensor(),T.Normalize((.5,.5,.5),(.5,.5,.5))])
train_ds=torchvision.datasets.CIFAR10('./data',train=True,download=True,transform=transform)
test_ds=torchvision.datasets.CIFAR10('./data',train=False,download=True,transform=transform)
train_loader=DataLoader(train_ds,batch_size=128,shuffle=True,num_workers=2)
test_loader=DataLoader(test_ds,batch_size=256,shuffle=False,num_workers=2)
classes=train_ds.classes
print(classes)

## 2. Inspect an image batch

In [ ]:
images,labels=next(iter(train_loader))
plt.imshow((images[0]*.5+.5).permute(1,2,0).clamp(0,1))
plt.title(classes[labels[0]])
plt.axis('off'); plt.show()

## 3. Define a compact CNN

Convolutions learn local patterns; pooling reduces spatial size.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.classifier=nn.Sequential(nn.Flatten(),nn.Linear(64*8*8,128),nn.ReLU(),nn.Dropout(.25),nn.Linear(128,10))
    def forward(self,x): return self.classifier(self.features(x))
model=SmallCNN().to(device)
loss_fn=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=.001)
print(model)

## 4. Train and evaluate

Start with two epochs to confirm the pipeline. Increase only if compute permits.

In [ ]:
def evaluate(loader):
    model.eval(); correct=total=0; loss_total=0.
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(device),y.to(device); logits=model(x); loss_total+=loss_fn(logits,y).item()*len(y)
            correct+=(logits.argmax(1)==y).sum().item(); total+=len(y)
    return loss_total/total,correct/total
history={'train_loss':[],'test_loss':[],'test_accuracy':[]}
for epoch in range(2):
    model.train(); total_loss=0.
    for x,y in train_loader:
        x,y=x.to(device),y.to(device); optimizer.zero_grad(); loss=loss_fn(model(x),y); loss.backward(); optimizer.step(); total_loss+=loss.item()*len(y)
    tl,acc=evaluate(test_loader); history['train_loss'].append(total_loss/len(train_ds)); history['test_loss'].append(tl); history['test_accuracy'].append(acc)
    print(f'Epoch {epoch+1}: train loss {history["train_loss"][-1]:.3f}, test loss {tl:.3f}, accuracy {acc:.3f}')

## 5. Plot learning curves

In [ ]:
plt.plot(history['train_loss'],label='Train loss'); plt.plot(history['test_loss'],label='Test loss'); plt.xlabel('Epoch'); plt.legend(); plt.show()
plt.plot(history['test_accuracy'],marker='o'); plt.xlabel('Epoch'); plt.ylabel('Test accuracy'); plt.show()

## Exercises

1. Increase filters from 32/64 to 64/128. Compare runtime.
2. Add a validation split and use it for model selection.
3. Describe signs of overfitting in the curves.
4. Why should the test set not guide repeated tuning?

This is a teaching baseline, not a high-accuracy CIFAR-10 solution.